In [ ]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    Dropout
)


In [2]:
data = [
    "i am a student",
    "how are you",
    "i love machine learning",
    "good morning",
    "thank you",
    "see you later",
    "what is your name",
    "where are you going",
    "i like coffee",
    "welcome"
]


In [3]:
# Add special tokens as plain words
bert_data = [
    "cls " + sentence + " sep"
    for sentence in data
]
vocab_size = 1000
sequence_length = 12

vectorizer = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
    standardize="lower"
)
# Include "mask" token in vocabulary
vectorizer.adapt(
    bert_data + ["mask"]
)


In [4]:

tokenized_sentences = vectorizer(
    bert_data
)

vocab = vectorizer.get_vocabulary()

mask_token_id = vocab.index("mask")

print("MASK TOKEN ID:", mask_token_id)
inputs = tokenized_sentences.numpy().copy()
targets = tokenized_sentences.numpy().copy()

np.random.seed(42)

for row in inputs:

    valid_positions = np.where(row != 0)[0]

    # Skip cls and sep positions
    if len(valid_positions) > 3:

        mask_position = np.random.choice(
            valid_positions[1:-1]
        )

        row[mask_position] = mask_token_id


MASK TOKEN ID: 16


In [5]:
# Positional Embedding


class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(
        self,
        sequence_length,
        vocab_size,
        embed_dim
    ):
        super().__init__()

        self.token_embedding = Embedding(
            vocab_size,
            embed_dim,
            mask_zero=True
        )

        self.position_embedding = Embedding(
            sequence_length,
            embed_dim
        )

    def call(self, inputs):

        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        token_embeddings = self.token_embedding(
            inputs
        )

        position_embeddings = self.position_embedding(
            positions
        )

        return token_embeddings + position_embeddings
use_causal_mask=False


In [6]:
class BERTBlock(tf.keras.layers.Layer):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ff_dim,
        dropout_rate=0.1
    ):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(
                ff_dim,
                activation="gelu"
            ),
            Dense(embed_dim)
        ])

        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

        self.dropout1 = Dropout(
            dropout_rate
        )

        self.dropout2 = Dropout(
            dropout_rate
        )

    def call(
        self,
        inputs,
        training=False
    ):

        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs
        )

        attention_output = self.dropout1(
            attention_output,
            training=training
        )

        x = self.norm1(
            inputs + attention_output
        )

        ffn_output = self.ffn(x)

        ffn_output = self.dropout2(
            ffn_output,
            training=training
        )

        return self.norm2(
            x + ffn_output)


In [7]:
# Build BERT


embed_dim = 64
num_heads = 2
ff_dim = 128
num_layers = 2

inputs_layer = tf.keras.Input(
    shape=(sequence_length,),
    dtype="int64"
)

x = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(inputs_layer)

for _ in range(num_layers):

    x = BERTBlock(
        embed_dim,
        num_heads,
        ff_dim
    )(x)

outputs = Dense(
    vocab_size,
    activation="softmax"
)(x)

bert_model = tf.keras.Model(
    inputs_layer,
    outputs
)

bert_model.summary()
bert_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

bert_model.fit(
    inputs,
    targets,
    epochs=200,
    batch_size=2,
    verbose=1
)
test_sentence = (
    "cls i love mask learning sep"
)

tokenized = vectorizer(
    [test_sentence]
)

predictions = bert_model.predict(
    tokenized,
    verbose=0
)

mask_position = np.where(
    tokenized[0].numpy() == mask_token_id
)[0][0]

predicted_id = np.argmax(
    predictions[
        0,
        mask_position
    ]
)

predicted_word = vocab[
    predicted_id
]

print("\nPrediction:")
print(predicted_word)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 12)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding            │ (None, 12, 64)         │        64,768 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bert_block (BERTBlock)          │ (None, 12, 64)         │        50,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bert_block_1 (BERTBlock)        │ (None, 12, 64)         │        50,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 12, 1000)       │        65,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 229,864 (897.91 KB)

 Trainable params: 229,864 (897.91 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - accuracy: 0.3667 - loss: 6.1073  
Epoch 2/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5833 - loss: 4.9497
Epoch 3/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6333 - loss: 4.3580
Epoch 4/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6667 - loss: 3.7935
Epoch 5/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7500 - loss: 3.2590
Epoch 6/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7667 - loss: 2.7646
Epoch 7/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7750 - loss: 2.3049
Epoch 8/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.7833 - loss: 1.9326
Epoch 9/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7833 - loss: 1.6140
Epoch 10/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8167 - loss: 1.3736
Epoch 11/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7917 - loss: 1.1791 
Epoch 12/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8500 